# **HMM0 · NEXT-OBSERVATION DISTRIBUTION**

In this example I can see the result of each toss, but the coin itself is hidden. 

### Q1. Specify π, A, and B for the coin model, using the state order (c1, c2) and observation order (H, T). Explain the meaning of one entry from each matrix or vector.

I use $(c_1,c_2)$ as the state order and $(H,T)$ for heads and tails. I define the matrices as:

$$
\pi=(0.5,0.5),\qquad
A=\begin{pmatrix}0.5&0.5\\0.5&0.5\end{pmatrix},\qquad
B=\begin{pmatrix}0.9&0.1\\0.5&0.5\end{pmatrix}.
$$

The first entry of $\pi$ means that the chance of starting with coin $c_1$ is 0.5. In $A$, the entry in row 1, column 2 means that after using $c_1$, the chance of choosing $c_2$ is 0.5. In $B$, the first row tells me that coin $c_1$ gives heads with probability 0.9 and tails with probability 0.1.

In [1]:
pi = [0.5, 0.5]
A = [[0.5, 0.5],
     [0.5, 0.5]]
B = [[0.9, 0.1],
     [0.5, 0.5]]

#print("P(heads with the second coin) =", B[1][0])

### Q2. For the coin model, use pt = (0.8, 0.2) and calculate ptA and ptAB. Why does multiplication by A produce (0.5, 0.5) regardless of the current state distribution? What property of A causes this, and what does each resulting vector represent?

I start with $p_t=(0.8,0.2)$. First, I multiply the current state distribution $p_t$ by the transition matrix $A$. This gives the distribution of the next hidden state:

$$
p_{t+1} = p_t A.
$$


$$
\begin{aligned}
p_{t+1}
&= p_t A \\
&= \begin{pmatrix} 0.8 & 0.2 \end{pmatrix}
   \begin{pmatrix} 0.5 & 0.5 \\ 0.5 & 0.5 \end{pmatrix} \\
&= \begin{pmatrix}
0.8 \cdot 0.5 + 0.2 \cdot 0.5 &
0.8 \cdot 0.5 + 0.2 \cdot 0.5
\end{pmatrix} \\
&= \begin{pmatrix} 0.5 & 0.5 \end{pmatrix}.
\end{aligned}
$$



In [2]:
#define a generic function that operates the moltiplication vectorxmatrix -> the assignment tell to NOT use NumPy
def vector_times_matrix(vector, matrix):
    result = []
    for j in range(len(matrix[0])):
        total = 0.0
        for i in range(len(vector)):
            total += vector[i] * matrix[i][j]
        result.append(total)
    return result

p_t = [0.8, 0.2]
next_state = vector_times_matrix(p_t, A)
print("Next coin [c1, c2] =", next_state)

Next coin [c1, c2] = [0.5, 0.5]


The vector $p_tA=(0.5,0.5)$ gives the probabilities for the next coin and both rows of A are the same, so the choice of the next coin is equally likely from either current state. This means that for any starting distribution $(p_1,p_2)$, the probability of each next coin is $0.5(p_1+p_2)=0.5$.

Then, I multiply the next state distribution by the emission matrix $B$, this gives the probability of the next emission. Since $p_{t+1} = p_t A$, I get:

$$
P(O_{t+1}) = p_{t+1}B = (p_t A)B = p_t AB.
$$


$$
\begin{aligned}
P(O_{t+1})
&= p_{t+1} B \\
&= \begin{pmatrix} 0.5 & 0.5 \end{pmatrix}
   \begin{pmatrix} 0.9 & 0.1 \\ 0.5 & 0.5 \end{pmatrix} \\
&= \begin{pmatrix}
0.5 \cdot 0.9 + 0.5 \cdot 0.5 &
0.5 \cdot 0.1 + 0.5 \cdot 0.5
\end{pmatrix} \\
&= \begin{pmatrix} 0.7 & 0.3 \end{pmatrix}.
\end{aligned}
$$

In [3]:
next_observation = vector_times_matrix(next_state, B)
print("Next result [H, T] =", next_observation)

Next result [H, T] = [0.7, 0.3]


## Reading the Kattis input

You will be given three matrices (in this order); transition matrix, emission matrix, and initial state probability distribution. The initial state probability distribution is a row vector encoded as a matrix with only one row. Each matrix is given on a separate line with the number of rows and columns followed by the matrix elements (ordered row by row). Note that the rows and column size can be different from the sample input.

In [4]:
#function that reads matrices form Kattis, It's applied to each line
def read_matrix(line):
    #split, the first two numbers are the number of rows and columns, next tokens are values 
    tokens = line.split()
    rows = int(tokens[0])
    cols = int(tokens[1])
    values = [float(token) for token in tokens[2:]]
    
    #add each row to the matrix, which goes from start (r*cols) to end (start+cols), where cols is read in input -> example: cols=3, [0*3,3],[1*3,6] etc
    matrix = []
    for r in range(rows): 
        start = r * cols
        end = start + cols
        row = values[start:end]
        matrix.append(row)
    return matrix


def hmm0(input_text):
    #extract lines and read all the matrices
    lines = input_text.strip().splitlines()
    transition = read_matrix(lines[0])
    emission = read_matrix(lines[1])
    initial = read_matrix(lines[2])[0]

    #calc as above p_t+1 and then O(t+1) vectors
    predicted_state = vector_times_matrix(initial, transition)
    predicted_observation = vector_times_matrix(predicted_state, emission)

    #from vectors to strings text
    return " ".join(["1", str(len(predicted_observation))] #rows(always 1) and col numbers 
                    + [str(value) for value in predicted_observation]) #values of predicted obs

### A test with two states and three possible observations

I chose a test where I can predict the answer by hand. The model starts in the first state and always moves to the second state on the next step. I therefore expect the result to match the second row of the emission matrix: 0.2, 0.3 and 0.5.

In [5]:
sample = """2 2 0 1 1 0
2 3 0.8 0.1 0.1 0.2 0.3 0.5
1 2 1 0"""

answer = hmm0(sample)
print(answer)
assert answer == "1 3 0.2 0.3 0.5"

1 3 0.2 0.3 0.5


The output starts with `1 3` because the result has one row and three columns. The last three numbers match my prediction and add up to 1.

